# Stage 0: Downsample Labels and Images

Downsample original labels and CT images by 2x in each dimension (1/8 volume) for faster processing.

**Input**: `labels/*.nii.gz`, `images/*.nii.gz`  
**Output**: `labels_ds/*.nii.gz`, `images_ds/*.nii.gz`

**v5 Change**: Keep largest connected component + second largest if within 0.3mm of the first.

All subsequent processing will use downsampled data. Final metrics will be scaled back to original resolution.

In [ ]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_correction_test_1.6.26"

# Downsample factor (2 = half resolution in each dimension)
DOWNSAMPLE_FACTOR = 2

In [ ]:
import sys
from pathlib import Path
import numpy as np
from scipy import ndimage
from scipy.ndimage import binary_erosion, binary_dilation, generate_binary_structure

sys.path.insert(0, str(Path('.').resolve()))
from utils import load_nifti, save_nifti, ensure_dir

# v5: Morphological cleaning parameters
ERODE_DILATE_ITERATIONS = 2

In [3]:
# Setup directories
target = Path(TARGET_DIR)
LABELS_INPUT_DIR = target / "labels"
IMAGES_INPUT_DIR = target / "images"
LABELS_OUTPUT_DIR = ensure_dir(target / "labels_ds")
IMAGES_OUTPUT_DIR = ensure_dir(target / "images_ds")

print(f"Labels input: {LABELS_INPUT_DIR}")
print(f"Labels output: {LABELS_OUTPUT_DIR}")
print(f"Images input: {IMAGES_INPUT_DIR}")
print(f"Images output: {IMAGES_OUTPUT_DIR}")
print(f"Downsample factor: {DOWNSAMPLE_FACTOR}x (1/{DOWNSAMPLE_FACTOR**3} volume)")

Labels input: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labels
Labels output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labels_ds
Images input: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/images
Images output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/images_ds
Downsample factor: 2x (1/8 volume)


In [4]:
def downsample_mask(mask, factor, affine):
    """Downsample a binary mask and adjust affine.
    
    Args:
        mask: Binary mask array
        factor: Downsample factor (2 = half resolution)
        affine: Original 4x4 affine matrix
    
    Returns:
        downsampled_mask: Smaller mask
        new_affine: Adjusted affine with scaled voxel size
    """
    # Use nearest-neighbor interpolation (order=0) to preserve binary values
    zoom_factor = 1.0 / factor
    downsampled = ndimage.zoom(mask.astype(np.float32), zoom_factor, order=0)
    downsampled = (downsampled > 0.5).astype(np.uint8)
    
    # Adjust affine: scale voxel size by factor
    new_affine = affine.copy()
    new_affine[:3, :3] *= factor  # Scale voxel dimensions
    
    return downsampled, new_affine


def downsample_image(image, factor, affine):
    """Downsample an intensity image and adjust affine.
    
    Args:
        image: Intensity image array
        factor: Downsample factor (2 = half resolution)
        affine: Original 4x4 affine matrix
    
    Returns:
        downsampled_image: Smaller image
        new_affine: Adjusted affine with scaled voxel size
    """
    # Use linear interpolation (order=1) for intensity images
    zoom_factor = 1.0 / factor
    downsampled = ndimage.zoom(image.astype(np.float32), zoom_factor, order=1)
    
    # Adjust affine: scale voxel size by factor
    new_affine = affine.copy()
    new_affine[:3, :3] *= factor  # Scale voxel dimensions
    
    return downsampled, new_affine

In [ ]:
# Process all label files
label_files = sorted(LABELS_INPUT_DIR.glob("*.nii.gz"))
print(f"Found {len(label_files)} label files to downsample")
print(f"v5: Applying erode({ERODE_DILATE_ITERATIONS}) then dilate({ERODE_DILATE_ITERATIONS}) to remove noise")
print("="*70)

# 3D structuring element for morphological operations
struct = generate_binary_structure(3, 1)  # 6-connectivity

# Store original shapes for upsampling reference
shape_info = []

for idx, label_file in enumerate(label_files, 1):
    sample_name = label_file.stem.replace('.nii', '')
    
    # Load original
    data, affine, header = load_nifti(label_file)
    mask = (data > 0).astype(np.uint8)
    original_voxels = np.sum(mask)
    original_shape = mask.shape
    
    # v5: Erode then dilate to remove noise/static
    mask_bool = mask.astype(bool)
    mask_eroded = binary_erosion(mask_bool, structure=struct, iterations=ERODE_DILATE_ITERATIONS)
    mask_cleaned = binary_dilation(mask_eroded, structure=struct, iterations=ERODE_DILATE_ITERATIONS)
    mask = mask_cleaned.astype(np.uint8)
    cleaned_voxels = np.sum(mask)
    removed_voxels = original_voxels - cleaned_voxels
    
    # Downsample
    ds_mask, ds_affine = downsample_mask(mask, DOWNSAMPLE_FACTOR, affine)
    ds_shape = ds_mask.shape
    ds_voxels = np.sum(ds_mask)
    
    # Save with same filename (no suffix change)
    output_file = LABELS_OUTPUT_DIR / label_file.name
    save_nifti(ds_mask, ds_affine, header, output_file)
    
    # Store shape info for later upsampling
    shape_info.append({
        'sample_name': sample_name,
        'original_shape': list(original_shape),
        'downsampled_shape': list(ds_shape),
        'original_affine': affine.tolist(),
    })
    
    removed_str = f" (removed {removed_voxels:,})" if removed_voxels > 0 else ""
    print(f"[{idx}/{len(label_files)}] {sample_name}: {original_shape} -> {ds_shape} ({original_voxels:,} -> {ds_voxels:,} voxels){removed_str}")

# Process all image files
print("\n" + "="*70)
image_files = sorted(IMAGES_INPUT_DIR.glob("*.nii.gz"))
print(f"Found {len(image_files)} image files to downsample")
print("="*70)

for idx, image_file in enumerate(image_files, 1):
    sample_name = image_file.stem.replace('.nii', '')
    
    # Load original
    data, affine, header = load_nifti(image_file)
    original_shape = data.shape
    
    # Downsample image (preserves intensity values)
    ds_image, ds_affine = downsample_image(data, DOWNSAMPLE_FACTOR, affine)
    ds_shape = ds_image.shape
    
    # Save with same filename
    output_file = IMAGES_OUTPUT_DIR / image_file.name
    save_nifti(ds_image.astype(np.float32), ds_affine, header, output_file)
    
    print(f"[{idx}/{len(image_files)}] {sample_name}: {original_shape} -> {ds_shape}")

In [6]:
# Save shape info for upsampling stage
import json
shape_info_file = ensure_dir(target / "metrics") / "downsample_info.json"
with open(shape_info_file, 'w') as f:
    json.dump({
        'downsample_factor': DOWNSAMPLE_FACTOR,
        'samples': shape_info
    }, f, indent=2)

# Summary
print("\n" + "="*70)
print("DOWNSAMPLE COMPLETE")
print("="*70)

label_output_files = list(LABELS_OUTPUT_DIR.glob("*.nii.gz"))
image_output_files = list(IMAGES_OUTPUT_DIR.glob("*.nii.gz"))

print(f"\nDownsampled labels: {len(label_output_files)} files -> {LABELS_OUTPUT_DIR}")
print(f"Downsampled images: {len(image_output_files)} files -> {IMAGES_OUTPUT_DIR}")
print(f"Shape info: {shape_info_file}")
print(f"\nVolume reduction: {DOWNSAMPLE_FACTOR**3}x (1/{DOWNSAMPLE_FACTOR**3} of original)")
print(f"\nAll subsequent stages will process downsampled data.")
print(f"Final metrics will be scaled: volumes x{DOWNSAMPLE_FACTOR**3}, lengths x{DOWNSAMPLE_FACTOR}")


DOWNSAMPLE COMPLETE

Downsampled labels: 30 files -> /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labels_ds
Downsampled images: 30 files -> /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/images_ds
Shape info: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/metrics/downsample_info.json

Volume reduction: 8x (1/8 of original)

All subsequent stages will process downsampled data.
Final metrics will be scaled: volumes x8, lengths x2
